<a href="https://www.kaggle.com/code/aabdollahii/9-ml-solution-final-dataset-pipline?scriptVersionId=339112975" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

path = "/kaggle/input/notebooks/aabdollahii/8-ml-solution-final-data-analysis/HVA_long_preprocessed.csv"
df = pd.read_csv(path)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nDtypes:")
print(df.dtypes)

print("\nFirst rows:")
display(df.head())

print("\nMissing values:")
display(df.isna().sum().to_frame("missing_count"))

print("\nBasic label distribution:")
display(df["label"].value_counts().rename(index={0: "human", 1: "machine"}).to_frame("count"))

print("\nBasic source distribution:")
display(df["source"].value_counts().to_frame("count"))

print("\nLabel percentage:")
display((df["label"].value_counts(normalize=True) * 100).rename(index={0: "human", 1: "machine"}).round(2).to_frame("percent"))

print("\nSource percentage:")
display((df["source"].value_counts(normalize=True) * 100).round(2).to_frame("percent"))


In [ ]:
required_cols = ["year", "filename", "word_count", "content", "source", "label", "n_words"]

missing_required = [col for col in required_cols if col not in df.columns]
print("Missing required columns:", missing_required)

df["content"] = df["content"].astype(str)
df["char_len"] = df["content"].str.len()
df["space_count"] = df["content"].str.count(" ")
df["sentence_punct_count"] = df["content"].str.count(r"[.!؟?]")
df["comma_count"] = df["content"].str.count(r"[,،]")
df["persian_digit_count"] = df["content"].str.count(r"[۰-۹]")
df["latin_digit_count"] = df["content"].str.count(r"[0-9]")
df["arabic_digit_count"] = df["content"].str.count(r"[٠-٩]")
df["newline_count"] = df["content"].str.count(r"[\n\r\t\v\f]")
df["zwnj_count"] = df["content"].str.count("\u200c")
df["latin_char_count"] = df["content"].str.count(r"[A-Za-z]")
df["arabic_ke_count"] = df["content"].str.count("ك")
df["arabic_ye_count"] = df["content"].str.count("ي")

print("Rows with newline/tab characters:", (df["newline_count"] > 0).sum())
print("Rows with English digits:", (df["latin_digit_count"] > 0).sum())
print("Rows with Arabic digits:", (df["arabic_digit_count"] > 0).sum())
print("Rows with Arabic ك:", (df["arabic_ke_count"] > 0).sum())
print("Rows with Arabic ي:", (df["arabic_ye_count"] > 0).sum())
print("Rows with Latin characters:", (df["latin_char_count"] > 0).sum())


In [ ]:
print("Within-filename source length comparison:")

pivot_words = df.pivot_table(
    index="filename",
    columns="source",
    values="n_words",
    aggfunc="mean"
)

display(pivot_words.head())

if set(["human", "qwen", "grok", "gpt"]).issubset(pivot_words.columns):
    pivot_words["qwen_minus_human"] = pivot_words["qwen"] - pivot_words["human"]
    pivot_words["grok_minus_human"] = pivot_words["grok"] - pivot_words["human"]
    pivot_words["gpt_minus_human"] = pivot_words["gpt"] - pivot_words["human"]

    display(pivot_words[["qwen_minus_human", "grok_minus_human", "gpt_minus_human"]].describe().round(2))

    diff_df = pivot_words[["qwen_minus_human", "grok_minus_human", "gpt_minus_human"]].reset_index()
    diff_long = diff_df.melt(id_vars="filename", var_name="comparison", value_name="word_difference")

    plt.figure(figsize=(10, 5))
    sns.boxplot(data=diff_long, x="comparison", y="word_difference")
    plt.axhline(0, color="black", linestyle="--", linewidth=1)
    plt.title("Machine Text Word Count Difference Compared with Human")
    plt.xlabel("Comparison")
    plt.ylabel("Word Difference")
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="source", order=df["source"].value_counts().index)
plt.title("Number of Texts by Source")
plt.xlabel("Source")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 5))
sns.countplot(data=df, x="label")
plt.title("Human vs Machine Label Distribution")
plt.xlabel("Label")
plt.ylabel("Count")
plt.xticks([0, 1], ["Human", "Machine"])
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data=df, x="n_words", hue="source", bins=50, kde=True, element="step")
plt.title("Word Count Distribution by Source")
plt.xlabel("Number of Words")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x="source", y="n_words")
plt.title("Word Count Boxplot by Source")
plt.xlabel("Source")
plt.ylabel("Number of Words")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
sns.violinplot(data=df, x="source", y="n_words", inner="quartile")
plt.title("Word Count Violin Plot by Source")
plt.xlabel("Source")
plt.ylabel("Number of Words")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data=df, x="char_len", hue="source", bins=50, kde=True, element="step")
plt.title("Character Length Distribution by Source")
plt.xlabel("Character Length")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x="source", y="char_len")
plt.title("Character Length Boxplot by Source")
plt.xlabel("Source")
plt.ylabel("Character Length")
plt.tight_layout()
plt.show()


In [ ]:
if "year" in df.columns:
    print("Year distribution:")
    display(df["year"].value_counts().sort_index().to_frame("count"))

    plt.figure(figsize=(12, 5))
    sns.countplot(data=df, x="year", hue="source")
    plt.title("Texts by Year and Source")
    plt.xlabel("Year")
    plt.ylabel("Count")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    year_source = pd.crosstab(df["year"], df["source"])
    display(year_source)

    plt.figure(figsize=(10, 6))
    sns.heatmap(year_source, annot=True, fmt="d", cmap="Blues")
    plt.title("Year x Source Frequency")
    plt.xlabel("Source")
    plt.ylabel("Year")
    plt.tight_layout()
    plt.show()


In [ ]:
print("Duplicate content analysis:")

exact_duplicate_rows = df.duplicated(subset=["content"]).sum()
print("Exact duplicate rows based on content:", exact_duplicate_rows)

duplicate_content_groups = (
    df.groupby("content")
    .agg(
        rows=("content", "count"),
        unique_sources=("source", "nunique"),
        unique_labels=("label", "nunique"),
        filenames=("filename", "nunique"),
    )
    .reset_index()
)

duplicate_content_groups = duplicate_content_groups[duplicate_content_groups["rows"] > 1]
duplicate_content_groups = duplicate_content_groups.sort_values("rows", ascending=False)

print("Number of duplicated content groups:", duplicate_content_groups.shape[0])
display(duplicate_content_groups.head(20))

possible_leakage = duplicate_content_groups[
    (duplicate_content_groups["unique_sources"] > 1) | 
    (duplicate_content_groups["unique_labels"] > 1)
]

print("Possible leakage duplicate groups across sources or labels:", possible_leakage.shape[0])
display(possible_leakage.head(20))


In [ ]:
print("Within-filename source length comparison:")

pivot_words = df.pivot_table(
    index="filename",
    columns="source",
    values="n_words",
    aggfunc="mean"
)

display(pivot_words.head())

if set(["human", "qwen", "grok", "gpt"]).issubset(pivot_words.columns):
    pivot_words["qwen_minus_human"] = pivot_words["qwen"] - pivot_words["human"]
    pivot_words["grok_minus_human"] = pivot_words["grok"] - pivot_words["human"]
    pivot_words["gpt_minus_human"] = pivot_words["gpt"] - pivot_words["human"]

    display(pivot_words[["qwen_minus_human", "grok_minus_human", "gpt_minus_human"]].describe().round(2))

    diff_df = pivot_words[["qwen_minus_human", "grok_minus_human", "gpt_minus_human"]].reset_index()
    diff_long = diff_df.melt(id_vars="filename", var_name="comparison", value_name="word_difference")

    plt.figure(figsize=(10, 5))
    sns.boxplot(data=diff_long, x="comparison", y="word_difference")
    plt.axhline(0, color="black", linestyle="--", linewidth=1)
    plt.title("Machine Text Word Count Difference Compared with Human")
    plt.xlabel("Comparison")
    plt.ylabel("Word Difference")
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()


In [ ]:
print("Text quality and preprocessing diagnostics:")

quality_summary = df.groupby("source").agg(
    rows=("content", "count"),
    rows_with_newline=("newline_count", lambda x: (x > 0).sum()),
    rows_with_latin_digits=("latin_digit_count", lambda x: (x > 0).sum()),
    rows_with_arabic_digits=("arabic_digit_count", lambda x: (x > 0).sum()),
    rows_with_persian_digits=("persian_digit_count", lambda x: (x > 0).sum()),
    rows_with_latin_chars=("latin_char_count", lambda x: (x > 0).sum()),
    rows_with_arabic_ke=("arabic_ke_count", lambda x: (x > 0).sum()),
    rows_with_arabic_ye=("arabic_ye_count", lambda x: (x > 0).sum()),
    mean_zwnj=("zwnj_count", "mean"),
    mean_sentence_punct=("sentence_punct_count", "mean"),
    mean_comma=("comma_count", "mean"),
).round(2)

display(quality_summary)


# ML pipeline - Using simple method 

In [ ]:
# =========================================
# Classic ML pipeline on already-preprocessed long dataset
# Human = 0, Grok/GPT/Qwen = 1
# No preprocessing
# Group split by filename to avoid leakage
# =========================================

import os
import warnings
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

warnings.filterwarnings("ignore")

# =========================================
# 1) Path
# =========================================

DATA_PATH = "/kaggle/input/notebooks/aabdollahii/8-ml-solution-final-data-analysis/HVA_long_preprocessed.csv"

print("Data path exists:", os.path.exists(DATA_PATH))
print("Data path:", DATA_PATH)

# =========================================
# 2) Load dataset
# =========================================

df = pd.read_csv(DATA_PATH)

print("\nDataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

display(df.head())

# =========================================
# 3) Basic checks
# =========================================

required_cols = ["filename", "source", "content"]

missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df = df.copy()

df["source"] = df["source"].astype(str).str.strip().str.lower()
df["content"] = df["content"].astype(str)

valid_sources = ["human", "grok", "gpt", "qwen"]

df = df[df["source"].isin(valid_sources)].reset_index(drop=True)

df["label"] = np.where(df["source"] == "human", 0, 1)

if "n_words" not in df.columns:
    df["n_words"] = df["content"].str.split().str.len()

if "n_chars" not in df.columns:
    df["n_chars"] = df["content"].str.len()

df = df[df["content"].str.len() > 0].reset_index(drop=True)

print("\nDataset shape after source filtering:", df.shape)

print("\nSource distribution:")
print(df["source"].value_counts())

print("\nLabel distribution:")
print(df["label"].value_counts().rename(index={0: "human", 1: "machine"}))

print("\nWord statistics by source:")
display(df.groupby("source")["n_words"].describe().round(2))

display(df.head(12))

# =========================================
# 4) Leakage check before splitting
# =========================================

print("\nFilename coverage:")

filename_coverage = (
    df.groupby("filename")
    .agg(
        rows=("content", "count"),
        unique_sources=("source", "nunique"),
        unique_labels=("label", "nunique")
    )
    .reset_index()
)

display(filename_coverage.head())

print("\nNumber of rows per filename:")
print(filename_coverage["rows"].value_counts().sort_index())

print("\nNumber of unique sources per filename:")
print(filename_coverage["unique_sources"].value_counts().sort_index())

incomplete_files = filename_coverage[filename_coverage["unique_sources"] < 4]

print("\nFilenames with fewer than 4 sources:", incomplete_files.shape[0])
display(incomplete_files.head(20))

# =========================================
# 5) Grouped train/test split
# =========================================

X = df["content"].values
y = df["label"].values
groups = df["filename"].values

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups=groups))

train_df = df.iloc[train_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

X_train_text = train_df["content"].values
y_train = train_df["label"].values

X_test_text = test_df["content"].values
y_test = test_df["label"].values

print("\nTrain shape:", train_df.shape)
print("Test shape :", test_df.shape)

print("\nTrain label distribution:")
print(train_df["label"].value_counts().rename(index={0: "human", 1: "machine"}))

print("\nTest label distribution:")
print(test_df["label"].value_counts().rename(index={0: "human", 1: "machine"}))

print("\nTrain source distribution:")
print(train_df["source"].value_counts())

print("\nTest source distribution:")
print(test_df["source"].value_counts())

train_files = set(train_df["filename"])
test_files = set(test_df["filename"])
overlap_files = train_files.intersection(test_files)

print("\nFilename overlap between train and test:", len(overlap_files))

if len(overlap_files) > 0:
    print("Warning: leakage risk. Some filenames appear in both train and test.")
else:
    print("No filename overlap. Group split is clean.")

# =========================================
# 6) Save train/test split
# =========================================

train_output_path = "/kaggle/working/HVA_train_split.csv"
test_output_path = "/kaggle/working/HVA_test_split.csv"

train_df.to_csv(train_output_path, index=False, encoding="utf-8-sig")
test_df.to_csv(test_output_path, index=False, encoding="utf-8-sig")

print("\nSaved train split:", train_output_path)
print("Saved test split :", test_output_path)

# =========================================
# 7) Models
# =========================================

def build_models():
    tfidf = TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
        lowercase=False
    )

    models = {
        "TFIDF+MultinomialNB": Pipeline([
            ("tfidf", clone(tfidf)),
            ("clf", MultinomialNB(alpha=1.0))
        ]),

        "TFIDF+LinearSVC": Pipeline([
            ("tfidf", clone(tfidf)),
            ("clf", LinearSVC(C=1.0, random_state=42))
        ]),

        "TFIDF+RandomForest": Pipeline([
            ("tfidf", clone(tfidf)),
            ("clf", RandomForestClassifier(
                n_estimators=300,
                max_depth=None,
                random_state=42,
                n_jobs=-1
            ))
        ]),

        "TFIDF+KNN": Pipeline([
            ("tfidf", clone(tfidf)),
            ("clf", KNeighborsClassifier(
                n_neighbors=15,
                metric="cosine"
            ))
        ]),
    }

    return models

# =========================================
# 8) Evaluation
# =========================================

def evaluate_model(model, X_train_text, y_train, X_test_text, y_test, name="model"):
    model.fit(X_train_text, y_train)

    y_pred = model.predict(X_test_text)

    y_score = None

    try:
        if hasattr(model, "predict_proba"):
            probs = model.predict_proba(X_test_text)
            if probs.ndim == 2 and probs.shape[1] == 2:
                y_score = probs[:, 1]
    except Exception:
        y_score = None

    try:
        if y_score is None and hasattr(model, "decision_function"):
            scores = model.decision_function(X_test_text)
            if np.ndim(scores) == 1:
                y_score = scores
    except Exception:
        y_score = None

    acc = accuracy_score(y_test, y_pred)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test,
        y_pred,
        average="binary",
        pos_label=1,
        zero_division=0
    )

    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

    roc_auc = np.nan

    if y_score is not None and len(np.unique(y_test)) == 2:
        try:
            roc_auc = roc_auc_score(y_test, y_score)
        except Exception:
            roc_auc = np.nan

    print(f"\n{'=' * 70}")
    print(name)
    print(f"{'=' * 70}")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-score : {f1:.4f}")

    if not np.isnan(roc_auc):
        print(f"ROC-AUC  : {roc_auc:.4f}")

    print("\nConfusion matrix:")
    cm_df = pd.DataFrame(
        cm,
        index=["true_human", "true_machine"],
        columns=["pred_human", "pred_machine"]
    )
    display(cm_df)

    print("\nClassification report:")
    print(classification_report(
        y_test,
        y_pred,
        labels=[0, 1],
        target_names=["human", "machine"],
        zero_division=0
    ))

    result = {
        "model": name,
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "tn": cm[0, 0],
        "fp": cm[0, 1],
        "fn": cm[1, 0],
        "tp": cm[1, 1],
    }

    pred_df = pd.DataFrame({
        "y_true": y_test,
        "y_pred": y_pred
    })

    if y_score is not None:
        pred_df["score_machine"] = y_score

    return result, pred_df, model

# =========================================
# 9) Run all models
# =========================================

models = build_models()

all_results = []
all_predictions = {}
trained_models = {}

for_model_error_check = []

for model_name, model in models.items():
    try:
        result, pred_df, trained_model = evaluate_model(
            model=model,
            X_train_text=X_train_text,
            y_train=y_train,
            X_test_text=X_test_text,
            y_test=y_test,
            name=model_name
        )

        all_results.append(result)
        all_predictions[model_name] = pred_df
        trained_models[model_name] = trained_model

    except Exception as e:
        print(f"\nError in model: {model_name}")
        print(e)
        for_model_error_check.append({
            "model": model_name,
            "error": str(e)
        })

results_df = pd.DataFrame(all_results)

if len(results_df) > 0:
    results_df = results_df.sort_values("f1", ascending=False).reset_index(drop=True)

print("\nFinal model comparison:")
display(results_df)

if len(for_model_error_check) > 0:
    error_df = pd.DataFrame(for_model_error_check)
    print("\nModel errors:")
    display(error_df)

# =========================================
# 10) Save results and predictions
# =========================================

results_output_path = "/kaggle/working/HVA_classic_ml_results.csv"
results_df.to_csv(results_output_path, index=False, encoding="utf-8-sig")

print("\nSaved results:", results_output_path)

for model_name, pred_df in all_predictions.items():
    safe_name = model_name.replace("+", "_").replace(" ", "_")

    pred_output_path = f"/kaggle/working/HVA_predictions_{safe_name}.csv"

    temp_test = test_df.copy()
    temp_test["y_true"] = pred_df["y_true"].values
    temp_test["y_pred"] = pred_df["y_pred"].values

    if "score_machine" in pred_df.columns:
        temp_test["score_machine"] = pred_df["score_machine"].values

    temp_test.to_csv(pred_output_path, index=False, encoding="utf-8-sig")

    print("Saved predictions:", pred_output_path)

if len(for_model_error_check) > 0:
    error_output_path = "/kaggle/working/HVA_model_errors.csv"
    pd.DataFrame(for_model_error_check).to_csv(error_output_path, index=False, encoding="utf-8-sig")
    print("Saved model errors:", error_output_path)
